In [1]:
path = "/home/andrew/Documents/wikiparse/extracted/AB/wiki_00"

In [4]:
with open(path, "r") as f:
    text = f.read() 
    
print(text)

<doc id="80897509" url="https://en.wikipedia.org/wiki?curid=80897509" title="Wallacea conspicua">
Wallacea conspicua

Wallacea conspicua is a species of &lt;a href="beetle"&gt;beetle&lt;/a&gt; in the family &lt;a href="Chrysomelidae"&gt;Chrysomelidae&lt;/a&gt;. It is found in Indonesia (&lt;a href="Sumatra"&gt;Sumatra&lt;/a&gt;).

</doc>



In [ ]:
from pathlib import Path


In [20]:
from re import compile
import re
from pydantic import BaseModel, Field
import html

class WikiDoc(BaseModel):
    id: int = Field(..., description="The unique identifier for the document.")
    url: str = Field(..., description="The URL of the document.")
    title: str = Field(..., description="The title of the document.")
    content: str = Field(..., description="The main content of the document.")

class WikiFile:
    def __init__(self, path: Path):
        self.path = path
        self.raw = self._load_text()
        self.docs = self.parse()

    def _load_text(self) -> str:
        with open(self.path, "r") as f:
            return f.read()
    
    def parse(self):
        # Placeholder for parsing logic
        # parse <doc id="<id>" url="<url> title="<title>"> ... </doc>
        reg = compile(
            r'<doc id="(?P<id>\d+)" url="(?P<url>[^"]+)" title="(?P<title>[^"]+)">(?P<content>.*?)</doc>', re.DOTALL
        )
        matches = reg.finditer(self.raw)
        docs = {}
        for match in matches:
            title = match.group("title")
            doc = WikiDoc(
                id=int(match.group("id")),
                url=match.group("url"),
                title=title,
                content=html.unescape(match.group("content").strip())
            )
            docs[doc.title] = doc
        return docs


In [21]:
file = WikiFile(Path(path))

In [23]:
print(file.docs)  # Print the first parsed document

{'Wallacea conspicua': WikiDoc(id=80897509, url='https://en.wikipedia.org/wiki?curid=80897509', title='Wallacea conspicua', content='Wallacea conspicua\n\nWallacea conspicua is a species of <a href="beetle">beetle</a> in the family <a href="Chrysomelidae">Chrysomelidae</a>. It is found in Indonesia (<a href="Sumatra">Sumatra</a>).')}
